# 두 데이터 합치기

> 파이썬 8강 · 요약과 통계

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [두 데이터 합치기](https://mioon1402.github.io/timeseriesdata/python/p08-merge.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/menu_orders.csv
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/weather.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. merge의 기본

**8-1. 두 파일 확인**

In [ ]:
import pandas as pd
sales = pd.read_csv("cafe_sales.csv", parse_dates=["date"])
weather = pd.read_csv("weather.csv", parse_dates=["date"])

print("매출 데이터:", sales.shape, list(sales.columns))
print("날씨 데이터:", weather.shape, list(weather.columns))
print()
weather.head(3)

**8-2. 붙여보기**

In [ ]:
합침 = sales.merge(weather[["date", "weather"]], on="date", how="left")

print("결과:", 합침.shape)
합침[["date", "visitors", "sales", "weather"]].head(3)

## 2. how — 네 가지 붙이기 방식

**8-3. 네 방식 비교**

In [ ]:
for 방식 in ["inner", "left", "right", "outer"]:
    결과 = sales.merge(weather[["date", "weather"]], on="date", how=방식)
    결측 = 결과["weather"].isna().sum()
    print(f"how='{방식}':{len(결과):>5}행   weather 결측 {결측:>3}개")

## 3. 행이 사라지는 사고 막기

**8-4. 행 수 점검 습관**

In [ ]:
before = len(sales)
합침 = sales.merge(weather[["date", "weather"]], on="date", how="left")
after = len(합침)

print(f"merge 전 {before}행 → 후 {after}행")
if before != after:
    print("⚠️ 행 수가 바뀌었습니다! 기준열에 중복이 있는지 확인하세요")
else:
    print("✓ 행 수 유지")

# 어느 날짜에 날씨가 없는지 확인
누락 = 합침[합침["weather"].isna()]
print(f"\n날씨가 없는 날: {len(누락)}일")
print(누락["date"].dt.strftime("%Y-%m-%d").head(5).tolist())

**8-5. indicator 로 출처 확인하기**

In [ ]:
검사 = sales.merge(weather[["date", "weather"]], on="date",
                  how="outer", indicator=True)

print(검사["_merge"].value_counts().to_string())

## 4. 행이 늘어나는 사고 (더 위험합니다)

**8-6. 중복 키가 만드는 폭발**

In [ ]:
# 일부러 중복이 있는 작은 표를 만들어본다
왼쪽 = pd.DataFrame({"날짜": ["1일", "2일"], "매출": [100, 200]})
오른쪽 = pd.DataFrame({"날짜": ["1일", "1일", "2일"], "메모": ["A", "B", "C"]})

print("왼쪽", len(왼쪽), "행 / 오른쪽", len(오른쪽), "행")
결과 = 왼쪽.merge(오른쪽, on="날짜", how="left")
print("합친 결과:", len(결과), "행 ← 늘어났습니다!")
print(결과.to_string(index=False))
print()
print("매출 합계:", 결과["매출"].sum(), "  (원래는", 왼쪽["매출"].sum(), ")")

**8-7. validate 로 자동 검사**

In [ ]:
# validate 를 주면 관계가 깨질 때 '조용히'가 아니라 '오류로' 알려준다
try:
    결과 = sales.merge(weather[["date", "weather"]], on="date",
                      how="left", validate="one_to_one")
    print("✓ 1:1 관계 확인됨 —", len(결과), "행")
except Exception as e:
    print("✗ 관계가 깨짐:", e)

## 5. 집계 후 붙이기

**8-8. 집계 → 병합**

In [ ]:
menu = pd.read_csv("menu_orders.csv", parse_dates=["date"])
print("메뉴 데이터:", menu.shape)
print(menu.head(3).to_string(index=False))

# 날짜별로 먼저 접는다 (6강의 groupby)
일별 = menu.groupby("date").agg(
    총주문=("orders", "sum"),
    메뉴종류=("menu", "nunique"),
    총액=("amount", "sum"),
).reset_index()

print("\n집계 후:", 일별.shape, "← 날짜당 한 행")

합침 = sales.merge(일별, on="date", how="left", validate="one_to_one")
print("병합 후:", 합침.shape)
print()
합침[["date", "visitors", "총주문", "메뉴종류"]].head(3)

## 6. concat — 위아래로 쌓기

**8-9. 세로로 쌓기**

In [ ]:
상반기 = sales[sales["date"].dt.month <= 6]
하반기 = sales[sales["date"].dt.month > 6]

print("상반기", len(상반기), "행 / 하반기", len(하반기), "행")

전체 = pd.concat([상반기, 하반기], ignore_index=True)
print("합친 결과:", len(전체), "행")
print("원본과 같은가:", len(전체) == len(sales))

## 7. 실전: 날씨가 매출에 미치는 영향

**8-10. 날씨별 방문객**

In [ ]:
합침 = sales.merge(weather[["date", "weather"]], on="date", how="left")

결과 = (합침.groupby("weather")
            .agg(일수=("visitors", "size"),
                 평균방문=("visitors", "mean"),
                 평균매출=("sales", "mean"))
            .round(1)
            .sort_values("평균방문"))
결과

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 날씨 데이터가 없는 23일이 특정 기간에 몰려 있는지 확인해보세요.
#        힌트: 누락된 날짜의 월별 분포를 세어보기


# 문제 2. 날씨별 '객단가'(매출 ÷ 방문객)를 구해보세요.
#        비 오는 날엔 사람이 적게 오는데, 온 사람은 더 쓸까요?


# 문제 3. menu_orders.csv 에서 카테고리(커피/라떼기타)별 월 매출을
#        pivot_table 로 만들어보세요.

**모범 답안**

In [ ]:
합침 = sales.merge(weather[["date", "weather"]], on="date", how="left")

# 문제 1
누락 = 합침[합침["weather"].isna()]
print("누락된 날짜의 월별 분포:")
print(누락["date"].dt.to_period("M").value_counts().sort_index().to_string())

# 문제 2
영업 = 합침[합침["visitors"] > 0].copy()
영업["객단가"] = 영업["sales"] / 영업["visitors"]
print()
print(영업.groupby("weather")["객단가"].mean().round(0).sort_values().to_string())

# 문제 3
menu = pd.read_csv("menu_orders.csv", parse_dates=["date"])
표 = menu.pivot_table(values="amount", index=menu["date"].dt.month,
                     columns="category", aggfunc="sum")
print()
(표 / 10000).round(0).head(6)

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)